## Koordinaattien käsittely ##

**co-pilotin kanssa keskusteltua, käsittely ja järjestys**
1. *Huomoidaan vain USA:ssa olevat pisteet ja rajataan sijoitetaan pisteet eri piirikuntiin*
- Koordinaatteihin perustuva on epätarkka. Alaska ja Hawaiji huomioitava myös
- Tehdään dataframesta geodataframe (gdf), mahdollistaa maantieteellisten sijaintien käsittelyn ja paikkatietoanalyysin tekemisen. Gdf perii kaikki df:n ominaisuudet, joten ei vaikuta jatkokäsittelyyn. 
- Tarjolla tiedosto, jossa sijainnit piirikuntien (county) tasolla. Tiedostoa voi hyödyntää analysoitaessa sijainnin vaikutusta hintoihin ja myyntiaikaan. Huomioitava että mukana myös USA:n itsehallintoalueet, ainakin Puerto Rico esiintyy autodatassa.
    - cb_2023_us_county_500k.shp
- tiedoston hyödyntäminen vaatii 
    - geopandas (gpd) kirjaston käyttämisen
    - df -> gdf, EPSG=mikä koordinaattijärjestelmä, WGS84 = 4326
    - gpd.sjoin -> jonka jälkeen jokaisella rivillä tieto missä auto myyty (myyjäliike) koordinaattien perusteella

*Miten piirikuntakohtaista dataa voi hyödyntää (ennustavan) mallin opetuksessa*
- lasketaan aggrekaattiarvot kts. alla, kaikille saman alueen riveille siis samat arvot
- aggrekaattiarvot kuvaavat alueen markkina ja taloudellisia ominaisuuksia
- Vaikka featuret ovat samanlaiset piirikuntien sisällä, ne eroavat voimakkaasti piirikuntien välillä — ja juuri tämä erottaa alueelliset markkinat toisistaan.

- näkemään millä alueella auto on
- oppimaan alueen markkinahinnan ja myyntinopeuden
- mallintamaan alueellisia eroja ilman 3000-sarakkeista one-hot-enkoodausta

- Toisin sanoen, aggregaattifeaturet tuovat alueellisen “ympäristön” datasi riveille.


*Aggrekaattiarvot lasketaan ainoastaan train datasta*
- Koska kaikki aggregaatit ovat target-dependent features, ja ne voivat sisältää vuotoa, jos ne lasketaan väärässä järjestyksessä. Kun aggregaatit lasketaan vain trainista:

- Testi pysyy puhtaana
- Malli ei näe testidatan hintoja etukäteen
- Mallin suorituskyky on realistinen
- Ennusteet toimivat käytännössä  




In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# -----------------------------------------------------------
# Ladataan projektin juuren .env-tiedosto.
# .env sisältää käyttäjäkohtaiset asetukset, joita EI jaeta Gitissä,
# kuten DATA_PATH = /oma/polku/data-kansioon/
# 
# Näin koodiin ei tarvitse kovakoodata tiedostopolkuja ja
# jokainen kehittäjä voi käyttää omaa paikallista data-sijaintiaan.
# -----------------------------------------------------------
load_dotenv()

# Luetaan datakansion polku .env-tiedostosta
DATA_PATH = Path(os.getenv("DATA_PATH"))

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

dtypes = {
    "daysonmarket": "int16",
    "engine_displacement": "float32",
    "horsepower": "float32",
    "latitude": "float32",
    "longitude": "float32",
    "mileage": "float32",
    "price": "float32",
    "year": "int16"
}

columns = [
    'body_type', 'daysonmarket','engine_displacement', 'engine_type','mileage',
    'fuel_type','horsepower','latitude','longitude','major_options',
    'make_name','model_name','price','transmission_display','wheel_system',
    'year'
]


df = pd.read_csv(DATA_PATH / "100_rivia_testiin", header=0, usecols=columns, dtype=dtypes)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

counties = gpd.read_file("cb_2023_us_county_500k.zip")


# Olennaiset sarakkeet counties-tiedostosta
cols = ["GEOID", "NAME", "STATEFP", "geometry"]

# Ladataan vain valitut sarakkeet
counties = counties[cols]

# CRS samaksi kuin pisteillä
counties = counties.to_crs(gdf.crs)

# Spatial join vain olennaisilla sarakkeilla
gdf = gpd.sjoin(
    gdf,
    counties,
    how="left",
    predicate="within"
)

# sjoin luo ylimääräisen 'index_right' -sarakkeen, poistetaan se
gdf = gdf.drop(columns="index_right")

gdf.tail(10)

,body_type,daysonmarket,engine_displacement,engine_type,fuel_type,horsepower,latitude,longitude,major_options,make_name,mileage,model_name,price,transmission_display,wheel_system,year,geometry,GEOID,NAME,STATEFP
90,Sedan,97,2000.0,I4,Gasoline,158.0,40.884701,-73.831703,"['Bluetooth', 'Backup Camera']",Honda,32894.0,Civic,14400.0,Continuously Variable Transmission,FWD,2017,POINT (-73.8317 40.8847),36005,Bronx,36
91,SUV / Crossover,24,2400.0,I4,Gasoline,180.0,40.733299,-73.258698,"['Alloy Wheels', 'Bluetooth', 'Backup Camera',...",Jeep,48522.0,Renegade,12924.0,6-Speed Manual,4WD,2015,POINT (-73.2587 40.7333),36103,Suffolk,36
92,Sedan,76,1500.0,I4,Gasoline,160.0,40.733299,-73.258698,"['Driver Confidence Package', 'Power Package',...",Chevrolet,61981.0,Malibu,14359.0,6-Speed Automatic,FWD,2018,POINT (-73.2587 40.7333),36103,Suffolk,36
93,SUV / Crossover,12,3500.0,V6,Gasoline,295.0,40.733299,-73.258698,"['Sport Package', 'Leather Seats', 'Sunroof/Mo...",Lexus,24021.0,RX 350,34949.0,8-Speed Automatic,AWD,2017,POINT (-73.2587 40.7333),36103,Suffolk,36
94,SUV / Crossover,19,2000.0,I4,Gasoline,241.0,40.884701,-73.831703,"['Leather Seats', 'Sunroof/Moonroof', 'Navigat...",Mercedes-Benz,81807.0,GLC-Class,22000.0,9-Speed Automatic,AWD,2016,POINT (-73.8317 40.8847),36005,Bronx,36
95,SUV / Crossover,70,2000.0,I4,Gasoline,246.0,18.443899,-66.078499,"['Adaptive Cruise Control', 'Backup Camera', '...",Land Rover,11.0,Range Rover Evoque,57535.0,9-Speed Automatic Overdrive,AWD,2020,POINT (-66.0785 18.4439),72127,San Juan,72
96,SUV / Crossover,77,3600.0,V6,Gasoline,281.0,40.733299,-73.258698,"['Power Package', 'Navigation System', 'Prefer...",Chevrolet,58971.0,Traverse,18959.0,6-Speed Automatic,AWD,2017,POINT (-73.2587 40.7333),36103,Suffolk,36
97,SUV / Crossover,70,3000.0,V6,Gasoline,340.0,18.443899,-66.078499,"['Backup Camera', 'Parking Sensors']",Land Rover,11.0,Range Rover Velar,77270.0,8-Speed Automatic Overdrive,AWD,2020,POINT (-66.0785 18.4439),72127,San Juan,72
98,Sedan,20,6200.0,V8,Gasoline,707.0,40.884701,-73.831703,"['Leather Seats', 'Navigation System', 'Alloy ...",Dodge,19652.0,Charger,57000.0,8-Speed Automatic,RWD,2018,POINT (-73.8317 40.8847),36005,Bronx,36
99,SUV / Crossover,187,3000.0,V6,Gasoline,340.0,18.443899,-66.078499,"['Leather Seats', 'Cargo Package', 'Comfort Pa...",Land Rover,11.0,Discovery,77885.0,8-Speed Automatic Overdrive,AWD,2020,POINT (-66.0785 18.4439),72127,San Juan,72


In [ ]:
def add_county_aggregates_train_test(train_df, test_df):

    train_df = train_df.copy()
    test_df = test_df.copy()

    # 1. hinta
    county_price_mean = train_df.groupby("COUNTYFP")["price"].mean()
    train_df["county_price_mean"] = train_df["COUNTYFP"].map(county_price_mean)
    test_df["county_price_mean"] = test_df["COUNTYFP"].map(county_price_mean)

    # 2. mediaani
    county_price_median = train_df.groupby("COUNTYFP")["price"].median()
    train_df["county_price_median"] = train_df["COUNTYFP"].map(county_price_median)
    test_df["county_price_median"] = test_df["COUNTYFP"].map(county_price_median)

    # 3. hajonta
    county_price_std = train_df.groupby("COUNTYFP")["price"].std()
    train_df["county_price_std"] = train_df["COUNTYFP"].map(county_price_std)
    test_df["county_price_std"] = test_df["COUNTYFP"].map(county_price_std)

    # 4. myyntiaika
    county_days_mean = train_df.groupby("COUNTYFP")["days_on_market"].mean()
    train_df["county_days_mean"] = train_df["COUNTYFP"].map(county_days_mean)
    test_df["county_days_mean"] = test_df["COUNTYFP"].map(county_days_mean)

    # 5. tarjonta
    county_supply = train_df.groupby("COUNTYFP")["id"].count()
    train_df["county_supply"] = train_df["COUNTYFP"].map(county_supply)
    test_df["county_supply"] = test_df["COUNTYFP"].map(county_supply)

    return train_df, test_df

In [ ]:
#tähän train test split ja mallin opetus. 